# Limpieza de datos

In [1]:
import pandas as pd

RAW = "../data/raw/"
PROC = "../data/processed/"

meses = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
         "agosto": 8, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}

## Tráfico vehicular ANI
Se dejan los peajes Las Palmas y Guarne y se suma el tráfico por mes.

In [2]:
ani = pd.read_csv(RAW + "trafico_ani.csv", dtype=str)
ani = ani[ani["Peaje"].isin(["LAS PALMAS", "GUARNE"])].copy()

ani["mes"] = pd.to_datetime(ani["Desde"], format="%Y %b %d %I:%M:%S %p")
ani["trafico"] = pd.to_numeric(ani["CantidadTrafico"].str.replace(",", ""), errors="coerce")
ani = ani.drop_duplicates(["IdPeaje", "CategoriaTarifa", "mes"])

trafico = ani.pivot_table(index="mes", columns="Peaje", values="trafico", aggfunc="sum")
trafico.columns = ["trafico_guarne", "trafico_las_palmas"]
trafico["trafico_total"] = trafico["trafico_guarne"] + trafico["trafico_las_palmas"]
trafico["trafico_livianos"] = ani[ani["CategoriaTarifa"] == "I"].groupby("mes")["trafico"].sum()
trafico = trafico.reset_index()
trafico.head()

,mes,trafico_guarne,trafico_las_palmas,trafico_total,trafico_livianos
0,2014-01-01,574186,223244,797430,581819
1,2014-02-01,455561,206812,662373,455323
2,2014-03-01,509916,233862,743778,519465
3,2014-04-01,514018,222521,736539,520308
4,2014-05-01,498184,229681,727865,498870


## DANE – Encuesta Mensual de Alojamiento (EMA)
Ocupación e índice de tarifas de la región Antioquia, desde el Excel de anexos.

In [3]:
archivo = RAW + "ema_anexos_sep2026.xlsx"

def leer_hoja_ema(hoja, columnas):
    d = pd.read_excel(archivo, sheet_name=hoja, header=None).iloc[12:, :].copy()
    d["anio"] = pd.to_numeric(d[0].astype(str).str.extract(r"(\d{4})")[0]).ffill()
    d["mes_num"] = d[1].astype(str).str.strip().str.lower().map(meses)
    d = d.dropna(subset=["anio", "mes_num"])
    out = pd.DataFrame({"mes": pd.to_datetime(dict(year=d["anio"].astype(int),
                                                   month=d["mes_num"].astype(int), day=1))})
    for nombre, col in columnas.items():
        out[nombre] = pd.to_numeric(d[col], errors="coerce").values
    return out

ocupacion = leer_hoja_ema("4.2 Porc Mens Ocupación.reg", {"ocupacion_ema_antioquia": 8})
tarifas = leer_hoja_ema("6.1 Ind.Var Tarifas.acomoda",
                        {"indice_tarifa_ema_sencilla": 38, "indice_tarifa_ema_doble": 41})
tarifas["indice_tarifa_ema_antioquia"] = tarifas[["indice_tarifa_ema_sencilla",
                                                  "indice_tarifa_ema_doble"]].mean(axis=1)
ema = ocupacion.merge(tarifas, on="mes", how="outer")
ema.head()

,mes,ocupacion_ema_antioquia,indice_tarifa_ema_sencilla,indice_tarifa_ema_doble,indice_tarifa_ema_antioquia
0,2019-01-01,46.537566,102.059113,105.041651,103.550382
1,2019-02-01,45.387830,96.163494,98.669385,97.416439
2,2019-03-01,44.436920,96.849923,99.557266,98.203594
3,2019-04-01,43.427767,98.451017,99.771105,99.111061
4,2019-05-01,44.480696,97.986075,96.493778,97.239926


## Migración Colombia
Entradas de extranjeros por el aeropuerto José María Córdova (se identifica por coordenadas), sumando todas las nacionalidades.

In [4]:
mig = pd.read_csv(RAW + "migracion_entradas_extranjeros.csv", dtype=str)
mig = mig[mig["Ubicación PCM"].isin(["(6.171601,-75.427454)", "(6.17134,-75.427194)"])].copy()

mig["Total"] = pd.to_numeric(mig["Total"].str.replace(",", ""), errors="coerce")
mig["mes_num"] = mig["Mes"].str.strip().str.lower().map(meses)
mig["mes"] = pd.to_datetime(dict(year=mig["Año"].astype(int), month=mig["mes_num"], day=1))

migracion = mig.groupby("mes", as_index=False)["Total"].sum()
migracion = migracion.rename(columns={"Total": "entradas_extranjeros_jmc"})
migracion.head()

,mes,entradas_extranjeros_jmc
0,2012-01-01,8967
1,2012-02-01,8522
2,2012-03-01,8853
3,2012-04-01,8204
4,2012-05-01,8906


## Unión mensual
Panel de 2019-01 a 2025-12 con las tres bases.

In [5]:
panel = pd.DataFrame({"mes": pd.date_range("2019-01-01", "2025-12-01", freq="MS")})
for base in [trafico, ema, migracion]:
    panel = panel.merge(base, on="mes", how="left")

panel.insert(1, "anio", panel["mes"].dt.year)
panel.insert(2, "mes_num", panel["mes"].dt.month)
panel["mes"] = panel["mes"].dt.strftime("%Y-%m")
panel.head()

,mes,anio,mes_num,trafico_guarne,trafico_las_palmas,trafico_total,trafico_livianos,ocupacion_ema_antioquia,indice_tarifa_ema_sencilla,indice_tarifa_ema_doble,indice_tarifa_ema_antioquia,entradas_extranjeros_jmc
0,2019-01,2019,1,749074,304798,1053872,785247,46.537566,102.059113,105.041651,103.550382,28360
1,2019-02,2019,2,593053,273802,866855,612713,45.387830,96.163494,98.669385,97.416439,29895
2,2019-03,2019,3,660017,315570,975587,703320,44.436920,96.849923,99.557266,98.203594,30289
3,2019-04,2019,4,659836,296791,956627,695277,43.427767,98.451017,99.771105,99.111061,26976
4,2019-05,2019,5,668668,318382,987050,700484,44.480696,97.986075,96.493778,97.239926,30958


## Verificación

In [6]:
inicio, fin = pd.Timestamp("2019-01-01"), pd.Timestamp("2025-12-01")
filas = []
for nombre, base in {"tráfico ANI": trafico, "EMA": ema, "migración": migracion}.items():
    en_periodo = base[(base["mes"] >= inicio) & (base["mes"] <= fin)]
    cruzan = en_periodo["mes"].dt.strftime("%Y-%m").isin(panel["mes"]).sum()
    filas.append({"base": nombre, "meses en la base": len(base),
                  "meses en 2019-2025": len(en_periodo), "cruzaron": cruzan})
display(pd.DataFrame(filas))

print("Filas del panel:", len(panel), "| columnas:", panel.shape[1])
print("Meses duplicados:", panel["mes"].duplicated().sum())
print("Valores faltantes:", panel.isna().sum().sum())

,base,meses en la base,meses en 2019-2025,cruzaron
0,tráfico ANI,149,84,84
1,EMA,93,84,84
2,migración,174,84,84


Filas del panel: 84 | columnas: 12
Meses duplicados: 0
Valores faltantes: 0


In [7]:
panel.to_csv(PROC + "panel_publico.csv", index=False)

## Aclaraciones
- La EMA es de Antioquia, no solo de Medellín, y su tarifa es un índice (base 2019 = 100), no pesos. Desde 2023 las cifras son provisionales.
- Migración cuenta entradas de extranjeros por el aeropuerto, no solo turistas.
- Los peajes se eligieron por estar en la ruta Medellín – aeropuerto; es un criterio del equipo.